### Data Analytics Project - Covid 19 Dataset

In [144]:
# Import libraries

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [145]:
# Import data
df_raw = pd.read_csv('../data/covid_19.csv')
df_raw.head(10)

,country,continent,population,day,time,Cases,Recovered,Deaths,Tests
0,Saint-Helena,Africa,6.115000e+03,2024-06-30,2024-06-30T16:15:16+00:00,2166,2.0,NaN,NaN
1,Falkland-Islands,South-America,3.539000e+03,2024-06-30,2024-06-30T16:15:16+00:00,1930,1930.0,NaN,8632.0
2,Montserrat,North-America,4.965000e+03,2024-06-30,2024-06-30T16:15:16+00:00,1403,1376.0,8.0,17762.0
3,Diamond-Princess,NaN,NaN,2024-06-30,2024-06-30T16:15:16+00:00,712,699.0,13.0,NaN
4,Vatican-City,Europe,7.990000e+02,2024-06-30,2024-06-30T16:15:16+00:00,29,29.0,NaN,NaN
5,Western-Sahara,Africa,6.261610e+05,2024-06-30,2024-06-30T16:15:16+00:00,10,9.0,1.0,NaN
6,MS-Zaandam,NaN,NaN,2024-06-30,2024-06-30T16:15:16+00:00,9,7.0,2.0,NaN
7,China,Asia,1.448471e+09,2024-06-30,2024-06-30T16:15:16+00:00,503302,379053.0,5272.0,160000000.0
8,Tokelau,Oceania,1.378000e+03,2024-06-30,2024-06-30T16:15:16+00:00,80,NaN,NaN,NaN
9,Saint-Pierre-Miquelon,North-America,5.759000e+03,2024-06-30,2024-06-30T16:15:16+00:00,3452,2449.0,2.0,25400.0


In [146]:
# Observe the data
df_raw.isnull().sum()

country        0
continent      2
population     9
day            0
time           0
Cases          0
Recovered     48
Deaths         5
Tests         25
dtype: int64

### We can see that there are multiple missing values on the data. We need to find strategy how to fill.
### My approach:
### 1. Continent and population: I'll remove the rows, since the it is name of a cruise
### 2. Recovered, Deaths and Tests: Median value (for the same region/continent)

In [147]:
# Observe the datatype
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 238 entries, 0 to 237
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   country     238 non-null    str    
 1   continent   236 non-null    str    
 2   population  229 non-null    float64
 3   day         238 non-null    str    
 4   time        238 non-null    str    
 5   Cases       238 non-null    int64  
 6   Recovered   190 non-null    float64
 7   Deaths      233 non-null    float64
 8   Tests       213 non-null    float64
dtypes: float64(4), int64(1), str(4)
memory usage: 16.9 KB


In [148]:
# We can change the day and time into string and datetime respectively
df_raw['day'] = pd.to_datetime(df_raw['day']).dt.day_name()
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 238 entries, 0 to 237
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   country     238 non-null    str    
 1   continent   236 non-null    str    
 2   population  229 non-null    float64
 3   day         238 non-null    str    
 4   time        238 non-null    str    
 5   Cases       238 non-null    int64  
 6   Recovered   190 non-null    float64
 7   Deaths      233 non-null    float64
 8   Tests       213 non-null    float64
dtypes: float64(4), int64(1), str(4)
memory usage: 16.9 KB


In [149]:
df_raw['time'] = pd.to_datetime(df_raw['time']).dt.normalize()
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 238 entries, 0 to 237
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype              
---  ------      --------------  -----              
 0   country     238 non-null    str                
 1   continent   236 non-null    str                
 2   population  229 non-null    float64            
 3   day         238 non-null    str                
 4   time        238 non-null    datetime64[us, UTC]
 5   Cases       238 non-null    int64              
 6   Recovered   190 non-null    float64            
 7   Deaths      233 non-null    float64            
 8   Tests       213 non-null    float64            
dtypes: datetime64[us, UTC](1), float64(4), int64(1), str(3)
memory usage: 16.9 KB


In [150]:
# We have removed the cruise
df_raw = df_raw.dropna(subset=['continent'])
df_raw.isnull().sum()

country        0
continent      0
population     7
day            0
time           0
Cases          0
Recovered     48
Deaths         5
Tests         23
dtype: int64

In [151]:
# We can remove the rows with null populations, since the data is not that clear
df_raw.loc[df_raw['population'].isnull()]
df_raw = df_raw.dropna(subset=['population'])

In [152]:
# Now we can fill the missing values on Recovered, Deaths and Tests with the median value of the specific continent or region
df_raw.loc[df_raw['Recovered'].isnull()].groupby('continent').count()

,country,population,day,time,Cases,Recovered,Deaths,Tests
continent,,,,,,,,
Africa,7,7,7,7,7,0,7,6
Asia,11,11,11,11,11,0,11,11
Europe,12,12,12,12,12,0,12,12
North-America,10,10,10,10,10,0,10,10
Oceania,6,6,6,6,6,0,5,2
South-America,2,2,2,2,2,0,2,2


In [153]:
median = df_raw.groupby('continent')['Recovered'].median()
median

continent
Africa             53569.0
Asia              727915.0
Europe           2004268.0
North-America      29805.0
Oceania             9563.0
South-America    1105645.0
Name: Recovered, dtype: float64

In [ ]:
# Fill the missing values
df_raw['Recovered'] =  df_raw['Recovered'].fillna(
    df_raw['continent'].map(median)
)

country        0
continent      0
population     0
day            0
time           0
Cases          0
Recovered      0
Deaths         5
Tests         16
dtype: int64

In [157]:
# Do the same thing with deaths and tests
death_median = df_raw.groupby('continent')['Deaths'].median()
death_median
df_raw['Deaths'] = df_raw['Deaths'].fillna(
    df_raw['continent'].map(death_median)
)

In [158]:
test_median =  df_raw.groupby('continent')['Tests'].median()
df_raw['Tests'] = df_raw['Tests'].fillna(
    df_raw['continent'].map(test_median)
)

In [ ]:
df_raw.columns = df_raw.columns.str.lower()
df_raw.columns

Index(['country', 'continent', 'population', 'day', 'time', 'cases',
       'recovered', 'deaths', 'tests'],
      dtype='str')

In [165]:
# Now we already have clean data can import to SQL for query or doing data exploration in python
df = df_raw.copy()

In [169]:
# Quick Statistical Insight
df.describe()

,population,cases,recovered,deaths,tests
count,2.290000e+02,2.290000e+02,2.290000e+02,2.290000e+02,2.290000e+02
mean,3.469404e+07,3.077525e+06,2.600729e+06,3.076972e+04,3.085712e+07
std,1.386374e+08,1.006101e+07,9.205055e+06,1.096416e+05,1.158633e+08
min,7.990000e+02,1.000000e+01,2.000000e+00,1.000000e+00,7.850000e+03
25%,4.454310e+05,2.733400e+04,2.980500e+04,2.250000e+02,3.059410e+05
50%,5.797805e+06,2.099060e+05,2.110800e+05,2.250000e+03,1.941032e+06
75%,2.210284e+07,1.356546e+06,1.503989e+06,1.492400e+04,1.171151e+07
max,1.448471e+09,1.118201e+08,1.098144e+08,1.219487e+06,1.186852e+09


In [ ]:
# Aggregation by Continent
aggregate_by_cont = df.groupby('continent')[['cases', 'recovered', 'deaths', 'tests']].sum()
# the most cases?
aggregate_by_cont.sort_values('cases', ascending=False).head(1)

,cases,recovered,deaths,tests
continent,,,,
Europe,253406198,259848390.0,2114042.0,2.859441e+09


In [196]:
# the most deaths
aggregate_by_cont.sort_values('deaths', ascending=False).head(1)  

,cases,recovered,deaths,tests
continent,,,,
Europe,253406198,259848390.0,2114042.0,2.859441e+09


In [198]:
# most test conducted
aggregate_by_cont.sort_values('tests', ascending=False).head(1)

,cases,recovered,deaths,tests
continent,,,,
Europe,253406198,259848390.0,2114042.0,2.859441e+09


In [ ]:
# Per capita metrics - we add new columns
df['cases_per_100k'] = df['cases']/df['population']*100000
df['deaths_per_100k'] = df['deaths']/df['population']*100000

In [ ]:
df['cases_per_100k'].sort_values(ascending=False).head(10)

KeyError: "None of [Index([ 77165.48691043057,  76822.64925920493,  70395.87268701887,\n        67352.31058997409,  67072.69781431192,  65289.76572133169,\n        65280.26009199079,  61984.43127686767,  61577.65439590256,\n       61201.273141932674],\n      dtype='float64')] are in the [columns]"